# Imagix3D
## Results check

Here, we are going to check out the results of Imagix3D pipeline runs.

In this particular case, we're looking at the **Computed Tomography Stroke Imaging** dataset. 

In [ ]:
# Parameters
RESULT_DIR = ""
PKL_PATH = ""
REPORT_DIR = ""

### Loading the data
**Please Note:** The results data are loaded in this notebook automatically by a SLURM job on the HPC (alpha). They cannot be loaded into RAM if the size of the results file surpasses your local working memory capacity (which is likely).  

In [ ]:
from pathlib import Path
import torch
import io
import os

import autoencodix as acx

RESULT_DIR = os.environ["RESULT_DIR"]
PKL_PATH = os.environ["PKL_PATH"]
REPORT_DIR = os.environ["REPORT_DIR"]

result_dir = Path(RESULT_DIR)
pkl_path = Path(PKL_PATH)
report_dir = Path(REPORT_DIR)

report_dir.mkdir(parents=True, exist_ok=True)

print("Result directory:", result_dir)
print("Pickle path:", pkl_path)
print("Report directory:", report_dir)

if not pkl_path.is_file():
    raise FileNotFoundError(f"Pickle file does not exist: {pkl_path}")

# Force CUDA tensors inside the pickle to be loaded on CPU
_original_load_from_bytes = torch.storage._load_from_bytes

def _load_from_bytes_cpu(bytes_object):
    return torch.load(io.BytesIO(bytes_object), map_location=torch.device("cpu"))

torch.storage._load_from_bytes = _load_from_bytes_cpu

try:
    imagix3d_loaded = acx.Imagix3D.load(str(pkl_path))
finally:
    torch.storage._load_from_bytes = _original_load_from_bytes

print("Loaded Imagix3D pipeline")

### Print dataset overview

In [ ]:
import pandas as pd

dataset = imagix3d_loaded.result.datasets

print("\nSample splits (in total):")
for split in ["train", "valid", "test"]:
    split_data = getattr(dataset, split, None)
    print(f"{split}: {len(split_data.sample_ids)} samples")

### Print VAE Model Output

In [ ]:
display(imagix3d_loaded.result.model)

### Print Overview of Config Settings

In [ ]:
import json
import pandas as pd
from IPython.display import display
from enum import Enum
from pathlib import Path

def make_json_safe(obj):
    """Recursively convert objects into JSON-friendly Python objects."""
    if isinstance(obj, Enum):
        return obj.value
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    return obj

def config_to_dict(config):
    """Convert Pydantic config object to a plain dictionary."""
    if hasattr(config, "model_dump"):  # Pydantic v2
        cfg = config.model_dump(mode="python")
    elif hasattr(config, "dict"):      # Pydantic v1
        cfg = config.dict()
    else:
        raise TypeError(f"Unsupported config type: {type(config)}")

    return make_json_safe(cfg)

cfg = config_to_dict(imagix3d_loaded.config)

print(json.dumps(cfg, indent=2, ensure_ascii=False))

### Print overview of Latent Dimension Values (and save them into the REPORT_DIR directory)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

result = imagix3d_loaded.result

# Use the notebook/report directory if already defined; otherwise use current directory.
REPORT_DIR = Path(globals().get("REPORT_DIR", Path.cwd())).resolve()
REPORT_DIR.mkdir(parents=True, exist_ok=True)

EPOCH = -1
SPLITS = ["train", "valid", "test"]


def as_2d_df(values, sample_ids=None, prefix="LatDim"):
    """Convert tensor/array/DataFrame latent-like values to a clean DataFrame."""
    if isinstance(values, pd.DataFrame):
        df = values.copy()
    else:
        if hasattr(values, "detach"):
            values = values.detach().cpu().numpy()

        arr = np.asarray(values)

        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)
        elif arr.ndim > 2:
            arr = arr.reshape(arr.shape[0], -1)

        df = pd.DataFrame(
            arr,
            columns=[f"{prefix}_{i}" for i in range(arr.shape[1])],
        )

    if sample_ids is not None and "sample_id" not in df.columns:
        if len(sample_ids) == len(df):
            df.insert(0, "sample_id", list(sample_ids))

    return df


def get_sample_ids(split):
    try:
        return result.sample_ids.get(split=split)
    except Exception:
        return None


latent_summaries = []

for split in SPLITS:
    sample_ids = get_sample_ids(split)

    # Latent values z
    latent_df = result.get_latent_df(epoch=EPOCH, split=split)
    latent_df.to_csv(
        REPORT_DIR / f"latent_values_{split}_last_epoch.csv",
        index=False,
    )

    # Mu values
    mu_values = result.mus.get(split=split, epoch=EPOCH)
    mu_df = as_2d_df(mu_values, sample_ids=sample_ids, prefix="Mu")
    mu_df.to_csv(
        REPORT_DIR / f"mu_values_{split}_last_epoch.csv",
        index=False,
    )

    # Sigma values as stored in result.sigmas
    sigma_values = result.sigmas.get(split=split, epoch=EPOCH)
    sigma_df = as_2d_df(sigma_values, sample_ids=sample_ids, prefix="Sigma")
    sigma_df.to_csv(
        REPORT_DIR / f"sigma_values_{split}_last_epoch.csv",
        index=False,
    )

    # Summary table for latent values
    latent_dim_cols = [
        col for col in latent_df.columns
        if str(col).startswith("LatDim_")
    ]

    if len(latent_dim_cols) == 0:
        latent_dim_cols = latent_df.select_dtypes(include=np.number).columns.tolist()

    summary = (
        latent_df[latent_dim_cols]
        .describe()
        .T
        .reset_index()
        .rename(columns={"index": "latent_dim"})
    )
    summary.insert(0, "split", split)
    latent_summaries.append(summary)


latent_summary = pd.concat(latent_summaries, ignore_index=True)
latent_summary.to_csv(
    REPORT_DIR / "latent_values_summary_last_epoch.csv",
    index=False,
)

display(latent_summary)

print(f"Saved latent, mu, sigma, and summary CSV files to:")
print(REPORT_DIR)

### Print overview of Loss Values (and save them into the REPORT_DIR directory)

In [ ]:
def loss_to_df(values, loss_name):
    """Convert loss values to a DataFrame."""
    if isinstance(values, pd.DataFrame):
        df = values.copy()
        if loss_name not in df.columns and df.shape[1] == 1:
            df.columns = [loss_name]
        return df.reset_index()

    if isinstance(values, pd.Series):
        return values.rename(loss_name).reset_index()

    if hasattr(values, "detach"):
        values = values.detach().cpu().numpy()

    arr = np.asarray(values).squeeze()

    if arr.ndim == 0:
        return pd.DataFrame(
            {
                "epoch": [EPOCH],
                loss_name: [float(arr)],
            }
        )

    return pd.DataFrame(
        {
            "epoch": np.arange(len(arr)),
            loss_name: arr,
        }
    )


def get_sub_loss_curve(key, split):
    """Get a sub-loss curve from result.sub_losses."""
    try:
        loss_store = result.sub_losses.get(key)
    except TypeError:
        loss_store = result.sub_losses.get(key=key)

    return loss_store.get(split=split)


loss_tables = []

for split in ["train", "valid"]:
    total_values = result.losses.get(split=split)
    recon_values = get_sub_loss_curve("recon_loss", split)
    var_values = get_sub_loss_curve("var_loss", split)

    total_df = loss_to_df(total_values, "total_loss")
    recon_df = loss_to_df(recon_values, "recon_loss")
    var_df = loss_to_df(var_values, "var_loss")

    loss_df = (
        total_df
        .merge(recon_df, on="epoch", how="outer")
        .merge(var_df, on="epoch", how="outer")
    )

    loss_df.insert(0, "split", split)
    loss_tables.append(loss_df)


losses_df = pd.concat(loss_tables, ignore_index=True)

losses_df.to_csv(
    REPORT_DIR / "losses_train_valid.csv",
    index=False,
)

display(losses_df.tail())

print(f"Saved loss CSV file to:")
print(REPORT_DIR / "losses_train_valid.csv")

### The Evaluate Step

Here, we use the `latent space` and compare it to other dimensionality reduction techniques like PCA, UMAP, or a random baseline.  
In the basic case, you can specify a column in your annotation data to use for a machine learning task, e.g., training a classifier on disease state or cancer type. We then use the representations (latent space, UMAP, etc.) to train this classifier/regressor.  
You have many options to customize this step for your needs, such as passing a scikit-learn model and defining evaluation metrics. Please refer to [3] for a complete guide.

[3] Tutorials/DeepDives/EvaluateTutorial.ipynb

In [ ]:
# In our annotation file, we have a column called "median_split"
# that refers to the median split of NIHSS scores of our patients.
imagix3d_loaded.evaluate(params=["median_split"])  

In [ ]:
# Next, we can visualize the evaluation output with:
import matplotlib.pyplot as plt
plt.close("all")
imagix3d_loaded.visualizer.show_evaluation(
    param="median_split",
    metric="roc_auc_ovo"
)

### Visualize the reconstructions as images


❗❗ **Run the next step first before you inspect pipeline visualizations** ❗❗  
In case there are matplotlib version differences between the HPC and the PC you inspect these results, stored plots might fail to load. In this case, you need to run 'visualize' first. Since doing so even if there are no matplotlib version issues does no harm, you might just run this step every time after loading the results and before checking visualizations of them. 

#### Comparing reconstructions to originals from a random selection

In [ ]:
imagix3d_loaded.visualizer.show_image_recon_grid(
    result=imagix3d_loaded.result, 
    n_samples=6
)

#### Comparing reconstructions to originals, wrt the best, median and worst examples

In [ ]:
imagix3d_loaded.visualizer.show_image_recon_grid(
    result=imagix3d_loaded.result,
    selection="bmw",
    metric="mse",
)

#### Quantitative reconstruction metrics per sample

Quantitative reconstruction metrics compare each original volume with its reconstructed version on a voxel-wise level. MSE, MAE, RMSE, and percentile errors summarize reconstruction accuracy, while comparisons of original and reconstructed intensity means/standard deviations indicate whether the model preserves global intensity and contrast structure.

In [ ]:
import numpy as np
import pandas as pd

def get_recon_epoch(result, split):
    if split == "test":
        return -1
    stored_epochs = [e for e in result.reconstructions._data.keys() if e != -1]
    return max(stored_epochs)

def compute_reconstruction_metrics(result, splits=("train", "valid", "test")):
    dataset = result.datasets
    rows = []

    for split in splits:
        split_data = getattr(dataset, split, None)
        if split_data is None:
            print(f"{split}: no dataset found")
            continue

        epoch = get_recon_epoch(result, split)
        recons = result.reconstructions.get(split=split, epoch=epoch)

        if recons is None:
            print(f"{split}: no reconstructions found for epoch {epoch}")
            continue

        n = min(len(recons), len(split_data.raw_data), len(split_data.sample_ids))
        print(f"{split}: computing reconstruction metrics for {n} samples")

        for idx in range(n):
            sample_id = split_data.sample_ids[idx]

            orig = np.asarray(split_data.raw_data[idx].img).squeeze()
            recon = np.asarray(recons[idx]).squeeze()

            abs_error = np.abs(orig - recon)
            sq_error = (orig - recon) ** 2

            rows.append({
                "split": split,
                "epoch": epoch,
                "sample_id": sample_id,
                "mse": float(np.mean(sq_error)),
                "mae": float(np.mean(abs_error)),
                "rmse": float(np.sqrt(np.mean(sq_error))),
                "p95_abs_error": float(np.percentile(abs_error, 95)),
                "p99_abs_error": float(np.percentile(abs_error, 99)),
                "max_abs_error": float(np.max(abs_error)),
                "orig_mean": float(np.mean(orig)),
                "recon_mean": float(np.mean(recon)),
                "orig_std": float(np.std(orig)),
                "recon_std": float(np.std(recon)),
            })

    return pd.DataFrame(rows)

recon_metrics = compute_reconstruction_metrics(imagix3d_loaded.result)
display(recon_metrics.head())
display(recon_metrics.groupby("split")[["mse", "mae", "rmse", "p95_abs_error", "orig_std", "recon_std"]].describe())

recon_metrics.to_csv(report_dir / "reconstruction_metrics_by_sample.csv", index=False)

#### Distribution of metrics

Summary of how reconstruction quality varies across patients and data splits.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=recon_metrics, x="split", y="mse", ax=ax)
sns.stripplot(data=recon_metrics, x="split", y="mse", ax=ax, alpha=0.4)
ax.set_title("Sample-wise reconstruction MSE by split")
ax.set_xlabel("Split")
ax.set_ylabel("MSE")
fig.tight_layout()
fig.savefig(report_dir / "reconstruction_mse_by_split.png", dpi=150, bbox_inches="tight")
plt.show()

#### How much do reconstructions vary across samples compared with the original images?

If recon_diversity_ratio ≈ 1, then reconstructions vary across samples about as much as original images do. <br>If recon_diversity_ratio << 1, then reconstructions are much less variable than the original images. This suggests that the model may be collapsing toward similar reconstructions for different samples. <br>If recon_diversity_ratio ≈ 0, then reconstructions are almost identical across samples. That would be bad. <br>If recon_diversity_ratio > 1, then reconstructions vary more than the originals. This could mean exaggerated noise, unstable decoding, or artifacts.

In [ ]:
def reconstruction_diversity_check(result, splits=("train", "valid", "test"), n_max=20):
    dataset = result.datasets
    rows = []

    for split in splits:
        split_data = getattr(dataset, split, None)
        if split_data is None:
            continue

        epoch = get_recon_epoch(result, split)
        recons = result.reconstructions.get(split=split, epoch=epoch)

        if recons is None:
            continue

        n = min(n_max, len(recons), len(split_data.raw_data))
        if n < 2:
            continue

        orig_stack = np.stack([
            np.asarray(split_data.raw_data[i].img).squeeze().astype(np.float32)
            for i in range(n)
        ])

        recon_stack = np.stack([
            np.asarray(recons[i]).squeeze().astype(np.float32)
            for i in range(n)
        ])

        orig_mean_voxel_std = float(orig_stack.std(axis=0).mean())
        recon_mean_voxel_std = float(recon_stack.std(axis=0).mean())

        # Also check whether consecutive reconstructions are almost identical.
        pairwise_recon_diffs = [
            float(np.mean(np.abs(recon_stack[i] - recon_stack[i + 1])))
            for i in range(n - 1)
        ]

        rows.append({
            "split": split,
            "epoch": epoch,
            "n_samples_used": n,
            "mean_voxel_std_originals": orig_mean_voxel_std,
            "mean_voxel_std_reconstructions": recon_mean_voxel_std,
            "recon_diversity_ratio": recon_mean_voxel_std / orig_mean_voxel_std if orig_mean_voxel_std != 0 else np.nan,
            "mean_abs_difference_between_consecutive_recons": float(np.mean(pairwise_recon_diffs)),
        })

    return pd.DataFrame(rows)

diversity_df = reconstruction_diversity_check(imagix3d_loaded.result, n_max=20)
display(diversity_df)

diversity_df.to_csv(report_dir / "reconstruction_diversity_check.csv", index=False)

#### Mean Original / Mean Reconstruction / Mean Error Maps

Mean original, mean reconstruction, and mean absolute error maps summarize reconstruction behavior at the complete dataset level. They show whether the model captures the average structure of the NCCT images and where reconstruction errors are spatially concentrated

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def compute_mean_original_recon_error(result, split="test", n_max=30):
    dataset = result.datasets
    split_data = getattr(dataset, split)
    epoch = get_recon_epoch(result, split)
    recons = result.reconstructions.get(split=split, epoch=epoch)

    n = min(n_max, len(recons), len(split_data.raw_data))

    orig_sum = None
    recon_sum = None
    error_sum = None

    for i in range(n):
        orig = np.asarray(split_data.raw_data[i].img).squeeze().astype(np.float32)
        recon = np.asarray(recons[i]).squeeze().astype(np.float32)
        error = np.abs(orig - recon)

        if orig_sum is None:
            orig_sum = np.zeros_like(orig, dtype=np.float64)
            recon_sum = np.zeros_like(recon, dtype=np.float64)
            error_sum = np.zeros_like(error, dtype=np.float64)

        orig_sum += orig
        recon_sum += recon
        error_sum += error

    return orig_sum / n, recon_sum / n, error_sum / n

mean_orig, mean_recon, mean_error = compute_mean_original_recon_error(
    imagix3d_loaded.result,
    split="test",
    n_max=30,
)

d_mid, h_mid, w_mid = [s // 2 for s in mean_orig.shape]

slices = [
    ("Mean original axial", mean_orig[d_mid, :, :], "gray"),
    ("Mean reconstruction axial", mean_recon[d_mid, :, :], "gray"),
    ("Mean absolute error axial", mean_error[d_mid, :, :], "magma"),
    ("Mean original coronal", mean_orig[:, h_mid, :], "gray"),
    ("Mean reconstruction coronal", mean_recon[:, h_mid, :], "gray"),
    ("Mean absolute error coronal", mean_error[:, h_mid, :], "magma"),
    ("Mean original sagittal", mean_orig[:, :, w_mid], "gray"),
    ("Mean reconstruction sagittal", mean_recon[:, :, w_mid], "gray"),
    ("Mean absolute error sagittal", mean_error[:, :, w_mid], "magma"),
]

fig, axes = plt.subplots(3, 3, figsize=(12, 12), constrained_layout=True)

for ax, (title, img, cmap) in zip(axes.ravel(), slices):
    ax.imshow(img, cmap=cmap, origin="lower")
    ax.set_title(title)
    ax.axis("off")

fig.suptitle("Mean original, reconstruction, and absolute error volumes", fontsize=14)
fig.savefig(report_dir / "mean_original_recon_error_test.png", dpi=150, bbox_inches="tight")
plt.show()

### Visualize Losses and Latent Spaces

In [ ]:
imagix3d_loaded.show_result()

In [ ]:
fig_loss_abs = imagix3d_loaded.visualizer.show_loss(plot_type="absolute") # Displays all loss components and factors over epochs
fig_loss_rel = imagix3d_loaded.visualizer.show_loss(plot_type="relative") # Displays contribution of loss components to total loss over epochs

#### Split-wise Latent Space Visualization

The split-wise latent-space visualization projects high-dimensional latent embeddings into two dimensions and colors samples by data split. It is used to check whether train, validation, and test samples occupy comparable regions of latent space or whether one split appears shifted or separated. Strong separation between splits may indicate distributional differences, preprocessing differences, or poor generalization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

result = imagix3d_loaded.result

stored_epochs = [e for e in result.latentspaces._data.keys() if e != -1]
final_epoch = max(stored_epochs)

latent_parts = []

for split in ["train", "valid", "test"]:
    epoch = -1 if split == "test" else final_epoch
    latent_df = result.get_latent_df(epoch=epoch, split=split)

    if latent_df is None or latent_df.empty:
        continue

    latent_numeric = latent_df.select_dtypes(include=[float, int]).copy()
    latent_numeric["split"] = split
    latent_numeric["sample_id"] = latent_df.index
    latent_parts.append(latent_numeric)

latent_all = pd.concat(latent_parts, axis=0, ignore_index=True)

X = latent_all.drop(columns=["split", "sample_id"]).to_numpy()
embedding = PCA(n_components=2).fit_transform(X)

latent_pca_df = pd.DataFrame({
    "PC1": embedding[:, 0],
    "PC2": embedding[:, 1],
    "split": latent_all["split"],
    "sample_id": latent_all["sample_id"],
})

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=latent_pca_df, x="PC1", y="PC2", hue="split", alpha=0.7, ax=ax)
ax.set_title("PCA of final latent embeddings, colored by split")
fig.tight_layout()
fig.savefig(report_dir / "latent_pca_by_split.png", dpi=150, bbox_inches="tight")
plt.show()

latent_pca_df.to_csv(report_dir / "latent_pca_by_split.csv", index=False)

#### Latent Norm and Reconstruction-error Relationship

Relates each sample’s latent-space (euclidean) norm to its reconstruction error. The latent norm measures how far a sample’s latent representation lies from the origin of the latent space, while reconstruction MSE measures how accurately the image is reconstructed. A positive relationship may indicate that latent-space outliers are harder for the decoder to reconstruct.

In [ ]:
latent_error_rows = []

for split in ["train", "valid", "test"]:
    epoch = -1 if split == "test" else final_epoch
    latent_df = result.get_latent_df(epoch=epoch, split=split)

    if latent_df is None or latent_df.empty:
        continue

    latent_numeric = latent_df.select_dtypes(include=[float, int])
    latent_norm = np.sqrt((latent_numeric ** 2).sum(axis=1))

    tmp = pd.DataFrame({
        "sample_id": latent_df.index,
        "split": split,
        "latent_norm": latent_norm.values,
    })

    latent_error_rows.append(tmp)

latent_norm_df = pd.concat(latent_error_rows, ignore_index=True)

latent_recon_df = recon_metrics.merge(
    latent_norm_df,
    on=["sample_id", "split"],
    how="inner",
)

display(latent_recon_df.head())

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=latent_recon_df,
    x="latent_norm",
    y="mse",
    hue="split",
    alpha=0.7,
    ax=ax,
)
ax.set_title("Reconstruction error vs latent norm")
ax.set_xlabel("Latent norm")
ax.set_ylabel("Reconstruction MSE")
fig.tight_layout()
fig.savefig(report_dir / "reconstruction_error_vs_latent_norm.png", dpi=150, bbox_inches="tight")
plt.show()

latent_recon_df.to_csv(report_dir / "latent_norm_reconstruction_error.csv", index=False)

## Correlations between latent dimensions

In [ ]:
imagix3d_loaded.visualizer.show_latent_corr(
    result=imagix3d_loaded.result,
    show_table=True,
    show_corrmat=True
)

### Train/Valid loss-gap table


The train/validation loss-gap table compares training and validation losses for each loss component across epochs. The absolute difference and ratio between validation and training loss quantify whether the model generalizes similarly to held-out data or whether a gap emerges during training. This might be useful for distinguishing stable learning, overfitting, and underfitting.

In [ ]:
import pandas as pd
import numpy as np

def loss_container_to_df(container):
    values = container.get()
    if not isinstance(values, dict):
        if hasattr(values, "to_dict"):
            values = values.to_dict()
        else:
            values = {i: val for i, val in enumerate(values)}
    return pd.DataFrame.from_dict(values, orient="index")

recon_df = loss_container_to_df(imagix3d_loaded.result.sub_losses.get(key="recon_loss"))
var_df = loss_container_to_df(imagix3d_loaded.result.sub_losses.get(key="var_loss"))
total_df = loss_container_to_df(imagix3d_loaded.result.losses)

loss_gap_rows = []

for term_name, df in [
    ("recon_loss", recon_df),
    ("var_loss", var_df),
    ("total_loss", total_df),
]:
    if {"train", "valid"}.issubset(df.columns):
        tmp = pd.DataFrame({
            "epoch": df.index,
            "display_epoch": df.index + 1,
            "loss_term": term_name,
            "train": df["train"].astype(float),
            "valid": df["valid"].astype(float),
        })
        tmp["valid_minus_train"] = tmp["valid"] - tmp["train"]
        tmp["valid_div_train"] = tmp["valid"] / tmp["train"].replace(0, np.nan)
        loss_gap_rows.append(tmp)

loss_gap_df = pd.concat(loss_gap_rows, ignore_index=True)
display(loss_gap_df.tail(20))

loss_gap_df.to_csv(report_dir / "loss_train_valid_gap.csv", index=False)

## Inspect latent dimension "activity"

First defined by Burda et al. (2015)[<sup>1</sup>](#fn1) and later taken up by Sinha & Dieng (2021)[<sup>2</sup>](#fn2) the **activity** of the dimension of the latent variables **z** is $Var(x) > \delta$ , where $\delta$  is a user-defined threshold set to 0.01 in the papers. An active dimension is termed an _active unit_ in Sinha & Dieng. If the variance of a given dimension is very low it indicates that this very dimension is not used / inactive. The higher the number of latent active units, the better the learned representations.

In [ ]:
from autoencodix.evaluate._imagix3d_evaluator import Imagix3DEvaluator

imagix3d_loaded.evaluator = Imagix3DEvaluator()

In [ ]:
imagix3d_loaded.result = imagix3d_loaded.evaluator.compute_latent_activity(
    result=imagix3d_loaded.result,
    threshold=0.1,
    splits=("train", "valid"),
    include_test=True,
)

imagix3d_loaded.visualizer.show_latent_activity(
    result=imagix3d_loaded.result,
    show_table=True,
    show_dim_table=False,
)

## Latent Traversal

Latent traversal is used here as a qualitative visualization technique to inspect what individual latent dimensions have learned to represent. Starting from a fixed latent vector, one latent dimension is systematically varied while all other dimensions are held constant, and the decoder generates the corresponding output images. If changing a specific latent dimension produces consistent changes in the reconstructed volume, this suggests that the dimension captures a meaningful factor of variation in the data.

**Compute latent traversal...**

In [ ]:
imagix3d_loaded.result = imagix3d_loaded.evaluator.compute_latent_traversal(
    result=imagix3d_loaded.result,
    latent_dims="top_kl",
    n_latent_dims=8,
    value_mode="percentile",
    n_values=5,
)

**...and show the results**

In [ ]:
imagix3d_loaded.visualizer.show_latent_traversal(
    result=imagix3d_loaded.result,
    view="axial",
)

In [ ]:
imagix3d_loaded.result = imagix3d_loaded.evaluator.compute_latent_traversal(
    result=imagix3d_loaded.result,
    latent_dims="top_kl",
    n_latent_dims=8,
    value_mode="prior",
    n_values=5,
    prior_range=(-2.0, 2.0),
)

In [ ]:
imagix3d_loaded.visualizer.show_latent_traversal(
    result=imagix3d_loaded.result,
    view="coronal",
)

**References**

<span id="fn1"><sup>1</sup> @misc{burda2016importanceweightedautoencoders,
      title={Importance Weighted Autoencoders}, 
      author={Yuri Burda and Roger Grosse and Ruslan Salakhutdinov},
      year={2016},
      eprint={1509.00519},
      archivePrefix={arXiv},
      primaryClass={cs.LG},
      url={https://arxiv.org/abs/1509.00519}, 
}</span> [↩] (#fn1)

<span id="fn2"><sup>2</sup> @misc{sinha2022consistencyregularizationvariationalautoencoders,
      title={Consistency Regularization for Variational Auto-Encoders}, 
      author={Samarth Sinha and Adji B. Dieng},
      year={2022},
      eprint={2105.14859},
      archivePrefix={arXiv},
      primaryClass={cs.LG},
      url={https://arxiv.org/abs/2105.14859}, 
}</span> [↩] (#fn2)